### 理论加速比
####  平均接受率： $\alpha$
* collect_logits.py: 分别填入两个模型路径，收集logits
* cal_accept_rate.py: 填入两个模型的logits路径，计算平均接受率
    遇到问题：计算平均值时，acc_i为1.0, 但acc_avg一直下降，打印结果并调试比对发现acc_sum+=无效不增加，推测可能是因为使用了float16格式，虽然不知道是为什么，但改为float之后正常了

#### core model: `core_moe.py`
`prefill_forward`: 完整推理模式，接收prefill输入，计算时更新expert_popularity 并 更新in gpu expert
一开始想直接用`mixtral_forward`作为core模式推理，接收完整的input_ids输入，但是无法复用完整模型推理生成的kv cache


##### draft: core_moe.py - test case

In [2]:
from draft_moe.core_moe import CoreMoE
import transformers
import torch
import numpy as np

model_path = "/zx_data1/models/mixtral/models--mistralai--Mixtral-8x7B-v0.1"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model = CoreMoE()

Loading checkpoint shards: 100%|██████████| 19/19 [00:13<00:00,  1.38it/s]


In [3]:
text = "What is the spiciest part of a chili pepper? The"
# text = "The Tower Building of the Little Rock Arsenal , also known as U.S. Arsenal Building , is a building located in MacArthur Park in downtown Little Rock , Arkansas ."
inputs = tokenizer(text, return_tensors="pt")
inputs_ids = inputs.input_ids.to(model.device)

input_tokens = tokenizer.convert_ids_to_tokens(inputs_ids[0].tolist())

n_tokens_decode = 1 # 以len - n_tokens_decode个token为prefill更新core
prefill_ids = inputs_ids[:, :-n_tokens_decode]
logits_ref = model.prefill_forward(
    prefill_ids,
    position_ids=torch.arange(prefill_ids.shape[0], device=model.device).unsqueeze(0),
)
print(f"prefill_ids: {prefill_ids}")
print(f"position_ids: {torch.arange(prefill_ids.shape[0], device=model.device).unsqueeze(0)}")
print(logits_ref)

logits = model.mixtral_forward(
    inputs_ids, 
    position_ids=torch.arange(inputs_ids.shape[0], device=model.device).unsqueeze(0),
)
print(logits)

for i in range(logits_ref.shape[1]):
    # i = logits_ref.shape[1] - 1
    logits_core = logits[0, i, :]
    logits_full = logits_ref[0, i, :]
    probs_core = torch.nn.functional.softmax(logits_core, dim=-1).cpu().tolist()
    probs_full = torch.nn.functional.softmax(logits_full, dim=-1).cpu().tolist()
    acc_rate = np.sum(np.minimum(probs_core, probs_full)) 
    print(f"Accuracy rate after token {input_tokens[i]}: {acc_rate:.4f}")


RuntimeError: CUDA error: CUBLAS_STATUS_ALLOC_FAILED when calling `cublasCreate(handle)`

In [17]:
text = "What is the spiciest part of a chili pepper? The"
# text = "The Tower Building of the Little Rock Arsenal , also known as U.S. Arsenal Building , is a building located in MacArthur Park in downtown Little Rock , Arkansas ."
inputs = tokenizer(text, return_tensors="pt")
inputs_ids = inputs.input_ids.to(model.device)

input_tokens = tokenizer.convert_ids_to_tokens(inputs_ids[0].tolist())

logits_ref = model.prefill_forward(
    inputs_ids,
    position_ids=torch.arange(inputs_ids.shape[0], device=model.device).unsqueeze(0),
)
print(logits_ref)

n_tokens_decode = 1 # 以len - n_tokens_decode个token为prefill更新core
prefill_ids = inputs_ids[:, :-n_tokens_decode]
model.prefill_forward(
    prefill_ids,
    position_ids=torch.arange(prefill_ids.shape[-1], device=model.device).unsqueeze(0),
)

logits = model.mixtral_forward(
    inputs_ids, 
    position_ids=torch.arange(inputs_ids.shape[-1], device=model.device).unsqueeze(0),
)
print(logits)

for i in range(logits_ref.shape[1]):
    logits_core = logits[0, i, :]
    logits_full = logits_ref[0, i, :]
    probs_core = torch.nn.functional.softmax(logits_core, dim=-1).cpu().tolist()
    probs_full = torch.nn.functional.softmax(logits_full, dim=-1).cpu().tolist()
    acc_rate = np.sum(np.minimum(probs_core, probs_full)) 
    print(f"Accuracy rate after token {input_tokens[i]}: {acc_rate:.4f}")

logits2 = logits
loc2 = model.expert_loc

tensor([[[ 5.5938,  5.5938,  9.1250,  ...,  5.9375,  5.0312,  6.3125],
         [ 1.1797,  1.1719, 11.8750,  ...,  0.9375,  5.7188,  5.1875],
         [ 1.7578,  1.7578, 11.9375,  ...,  2.7969,  5.4688,  4.7500],
         ...,
         [ 1.0547,  1.0469, 12.6875,  ...,  0.1191,  3.6562,  3.9219],
         [ 3.4688,  3.4531, 18.2500,  ...,  4.0000,  6.2188,  5.8125],
         [ 2.5000,  2.5000, 13.1250,  ...,  3.9375,  5.9688,  6.2812]]],
       device='cuda:3', dtype=torch.bfloat16)
tensor([[[ 6.5312,  6.5312,  6.4375,  ...,  6.0625,  2.0469,  4.8125],
         [ 1.4844,  1.4844, 16.6250,  ...,  2.0938,  4.4375,  4.2812],
         [ 3.0469,  3.0469, 19.2500,  ...,  2.7188,  3.4219,  3.6406],
         ...,
         [ 0.5352,  0.5312, 19.3750,  ...,  1.5000,  3.6250,  4.5000],
         [ 2.3125,  2.2969, 21.6250,  ...,  4.3438,  4.2812,  4.8438],
         [ 2.4844,  2.4844, 17.8750,  ...,  2.8906,  4.3125,  4.4375]]],
       device='cuda:3', dtype=torch.bfloat16)
Accuracy rate after toke

In [2]:
text = "What is the spiciest part of a chili pepper? The"
# text = "The Tower Building of the Little Rock Arsenal , also known as U.S. Arsenal Building , is a building located in MacArthur Park in downtown Little Rock , Arkansas ."
inputs = tokenizer(text, return_tensors="pt")
inputs_ids = inputs.input_ids.to(model.device)
position_ids = torch.arange(
    0, inputs_ids.shape[-1], dtype=torch.long, device=model.device
).unsqueeze(0).view(-1, inputs_ids.shape[-1])

logits_input = model.prefill_forward(
    inputs_ids,
    position_ids,
)
n_tokens_decode = 1
print(logits_input)
logits_prefill = model.prefill_forward(
    inputs_ids[:, :-n_tokens_decode],
    position_ids=position_ids[:, :-n_tokens_decode],
)
print(logits_prefill)
print(f"inputs_ids: {inputs_ids}")
print(f"position_ids: {position_ids}")
print(f"prefill_ids: {inputs_ids[:, :-n_tokens_decode]}")
print(f"position_ids: {position_ids[:, :-n_tokens_decode]}")

tensor([[[ 2.5938,  2.5938,  9.1875,  ...,  5.8750,  3.1094,  3.0781],
         [ 0.7695,  0.7617,  6.2188,  ...,  2.1094,  6.3125,  6.2188],
         [ 4.1562,  4.1562, 14.1250,  ...,  5.5000,  6.7188,  6.0312],
         ...,
         [ 1.8672,  1.8594, 18.2500,  ...,  1.8203,  1.0156,  3.4219],
         [ 3.7500,  3.7344, 19.2500,  ...,  4.6562,  4.7812,  4.9688],
         [ 2.4531,  2.4531, 13.6875,  ...,  2.2656,  4.7812,  4.8438]]],
       device='cuda:3', dtype=torch.bfloat16)
tensor([[[ 2.5938,  2.5938,  9.1875,  ...,  5.8750,  3.1094,  3.0781],
         [ 0.7695,  0.7617,  6.2188,  ...,  2.1094,  6.3125,  6.2188],
         [ 4.1562,  4.1562, 14.1250,  ...,  5.5000,  6.7188,  6.0312],
         ...,
         [ 0.5508,  0.5469, 14.9375,  ...,  0.4219,  2.5781,  5.5938],
         [ 1.8438,  1.8281, 18.2500,  ...,  1.8125,  1.0156,  3.4219],
         [ 3.6406,  3.6250, 19.2500,  ...,  4.5625,  4.6875,  4.8750]]],
       device='cuda:3', dtype=torch.bfloat16)
inputs_ids: tensor([[   

In [2]:
text = "What is the spiciest part of a chili pepper? The"
# text = "The Tower Building of the Little Rock Arsenal , also known as U.S. Arsenal Building , is a building located in MacArthur Park in downtown Little Rock , Arkansas ."
inputs = tokenizer(text, return_tensors="pt")
inputs_ids = inputs.input_ids.to(model.device)
position_ids = torch.arange(
    0, inputs_ids.shape[-1], dtype=torch.long, device=model.device
).unsqueeze(0).view(-1, inputs_ids.shape[-1])

# inputs_ids对应的token
print(f"Input tokens: {tokenizer.convert_ids_to_tokens(inputs_ids[0].tolist())}")

n_tokens_decode = 3 # 以len - n_tokens_decode个token为prefill更新core

logits_ref = model.prefill_forward(inputs_ids, position_ids)[:, -n_tokens_decode:, :]
tokens_ref = tokenizer.convert_ids_to_tokens(inputs_ids[0, -n_tokens_decode:].tolist())

prefill_ids = inputs_ids[:, :-n_tokens_decode]
prefill_position_ids = position_ids[:, :-n_tokens_decode]
decode_ids = inputs_ids[:, -n_tokens_decode:]
decode_position_ids = position_ids[:, -n_tokens_decode:]

model.prefill_forward(prefill_ids, prefill_position_ids)
logits = model.decode_forward(decode_ids, decode_position_ids)
# print(logits)

for i in range(logits_ref.shape[1]):
    # i = logits_ref.shape[1] - 1
    logits_core = logits[0, i, :]
    logits_full = logits_ref[0, i, :]
    probs_core = torch.nn.functional.softmax(logits_core, dim=-1).cpu().tolist()
    probs_full = torch.nn.functional.softmax(logits_full, dim=-1).cpu().tolist()
    acc_rate = np.sum(np.minimum(probs_core, probs_full)) 
    print(f"Accuracy rate after token {tokens_ref[i]}: {acc_rate:.4f}")

Input tokens: ['<s>', '▁What', '▁is', '▁the', '▁sp', 'ici', 'est', '▁part', '▁of', '▁a', '▁ch', 'ili', '▁pepper', '?', '▁The']


Accuracy rate after token ▁pepper: 0.5179
Accuracy rate after token ?: 0.3226
Accuracy rate after token ▁The: 0.7302


##### draft: core_moe.py - prefill_forward & update_experts_loc


In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import transformers

model_path = "/zx_data1/models/mixtral/models--mistralai--Mixtral-8x7B-v0.1"

model = transformers.AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    ).eval()
lm_head = model.lm_head
model = model.model
n_layer = len(model.layers)
n_expert = len(model.layers[0].block_sparse_moe.experts)

/opt/conda/envs/fiddler/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 19/19 [00:17<00:00,  1.10it/s]


In [ ]:
n_tokens_decode = 1

tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
text = "What is the spiciest part of a chili pepper? The"
inputs = tokenizer(text, return_tensors="pt")
inputs_ids = inputs.input_ids.to(model.device)

position_ids = torch.arange(
    0, inputs_ids.shape[-1], dtype=torch.long, device=model.device
)
position_ids = position_ids.unsqueeze(0).view(-1, inputs_ids.shape[-1])

prefill_ids = inputs_ids[:, :-n_tokens_decode]
prefill_position_ids = position_ids[:, :-n_tokens_decode]
decode_ids = inputs_ids[:, -n_tokens_decode:]
decode_position_ids = position_ids[:, -n_tokens_decode:]
# print(f"prefill_ids: {prefill_ids}, decode_ids: {decode_ids}")
# print(f"prefill_position_ids: {prefill_position_ids}, decode_position_ids: {decode_position_ids}")

past_key_value = transformers.cache_utils.DynamicCache.from_legacy_cache()
expert_popularity = np.zeros(
    (n_layer, n_expert), dtype=float
)

hidden_dim = model.config.hidden_size


prefill_ids: tensor([[    1,  1824,   349,   272,   668,  2650,   374,   744,   302,   264,
           484,  2689, 19082, 28804]], device='cuda:0'), decode_ids: tensor([[415]], device='cuda:0')
prefill_position_ids: tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13]],
       device='cuda:0'), decode_position_ids: tensor([[14]], device='cuda:0')


In [8]:
with torch.no_grad():
    inps = model.embed_tokens(prefill_ids)

    for i_layer, layer in enumerate(model.layers):
        original_inps_shape = inps.shape
        inps_residual = inps
        inps = layer.input_layernorm(inps)
        inps, self_attn_weights, present_key_value = layer.self_attn(
            inps,
            position_ids=prefill_position_ids,
            past_key_value=past_key_value,
            use_cache=True,
        )

        inps = inps + inps_residual.to(inps.device)
        inps_residual = inps
        inps = layer.post_attention_layernorm(inps)
        inps = inps.view(-1, hidden_dim)
        router_logits = layer.block_sparse_moe.gate(inps)
        # routing_weights.shape: (batch_size * seq_len, num_experts)
        routing_weights = F.softmax(router_logits, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)

        expert_popularity[i_layer] = routing_weights.cpu().sum(axis=0).float().numpy() / routing_weights.shape[0]
        # print(
        #     f"Layer {i_layer + 1}/{n_layer}, "
        #     f"Expert popularity: {expert_popularity[i_layer]}"
        #     f"sum of Expert popularity: {expert_popularity[i_layer].sum()}"
        # )
        routing_weights, selected_experts = torch.topk(routing_weights, 2, dim=-1)

        inps_after_experts = torch.zeros_like(inps, device=model.device)
        experts = layer.block_sparse_moe.experts
        expert_mask = torch.nn.functional.one_hot(
            selected_experts, num_classes=n_expert
        ).permute(2, 1, 0)
        for i_expert in range(len(experts)):
            top_i, token_i = torch.where(expert_mask[i_expert])
            if(token_i.shape[0] == 0):
                    continue
            token_i_list = token_i.tolist()
            top_i_list = top_i.tolist()

            current_state = inps[None, token_i_list].reshape(-1, hidden_dim)
            current_state = experts[i_expert](
                current_state, routing_weights[token_i_list, top_i_list, None]
            )
            inps_after_experts.index_add_(
                0, token_i.to(inps_after_experts.device), current_state.to(inps.dtype).to(inps_after_experts.device)
            )
        
        inps = inps_residual.to(inps.device) + inps_after_experts.reshape(original_inps_shape).to(inps.device)


In [ ]:
''' update_expert_loc '''
expert_loc = np.zeros(
    (n_layer, n_expert), dtype=int
)
# (start_layer, end_layer)
start_layer = 2
end_layer = 30

for i_layer in range(0, start_layer):
    expert_loc[i_layer] = np.ones(n_expert, dtype=int)

for i_layer in range(end_layer, n_layer):
    expert_loc[i_layer] = np.ones(n_expert, dtype=int)

for i_layer in range(start_layer, end_layer):
    # print(f"expert popularity for layer {i_layer}: {expert_popularity[i_layer]}\n")
    _, on_gpu_experts = torch.topk(
        torch.tensor(expert_popularity[i_layer]),
        k=2,
    )
    
    expert_loc[i_layer] = torch.nn.functional.one_hot(
        on_gpu_experts, num_classes=n_expert
    ).sum(dim=0).cpu().numpy()

    # print(f"expert location for layer {i_layer}: {expert_loc[i_layer]}")
print(f"expert location: {expert_loc}")

In [10]:
with torch.no_grad():
    hidden_dim = model.config.hidden_size
    inps = model.embed_tokens(decode_ids)
    for i_layer, layer in enumerate(model.layers):
        original_inps_shape = inps.shape
        inps_residual = inps
        inps = layer.input_layernorm(inps)
        inps, self_attn_weights, present_key_value = layer.self_attn(
            inps,
            position_ids=decode_position_ids,
            past_key_value=past_key_value,
            use_cache=True,
        )

        inps = inps + inps_residual.to(inps.device)
        inps_residual = inps
        inps = layer.post_attention_layernorm(inps)
        inps = inps.view(-1, hidden_dim)
        router_logits = layer.block_sparse_moe.gate(inps)
        routing_weights = F.softmax(router_logits, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)

        expert_loc_i = torch.tensor(
             expert_loc[i_layer], 
             device=routing_weights.device,
             dtype=torch.bfloat16
        ).unsqueeze(0).repeat(routing_weights.shape[0], 1)
        routing_weights = routing_weights * expert_loc_i
        routing_weights, selected_experts = torch.topk(routing_weights, 2, dim=-1)

        inps_after_experts = torch.zeros_like(inps, device=model.device)
        experts = layer.block_sparse_moe.experts
        expert_mask = torch.nn.functional.one_hot(
            selected_experts, num_classes=n_expert
        ).permute(2, 1, 0)
        
        for i_expert in range(len(experts)):
            if expert_loc[i_layer][i_expert] == 0:
                continue
            top_i, token_i = torch.where(expert_mask[i_expert])
            if(token_i.shape[0] == 0):
                    continue
            token_i_list = token_i.tolist()
            top_i_list = top_i.tolist()

            current_state = inps[None, token_i_list].reshape(-1, hidden_dim)
            current_state = experts[i_expert](
                current_state, routing_weights[token_i_list, top_i_list, None]
            )
            inps_after_experts.index_add_(
                0, token_i.to(inps_after_experts.device), current_state.to(inps.dtype).to(inps_after_experts.device)
            )
        
        inps = inps_residual.to(inps.device) + inps_after_experts.reshape(original_inps_shape).to(inps.device)

    inps = model.norm(inps)
    lm_logits = lm_head(inps)
    print(lm_logits)

tensor([[[ 2.8906,  2.8594, 17.7500,  ...,  2.8438,  4.1875,  4.5625]]],
       device='cuda:3', dtype=torch.bfloat16)





# drafts

In [9]:
# tokenizer = transformers.AutoTokenizer.from_pretrained(config.model)
text = "What is the spiciest part of a chili pepper? The"
inputs = tokenizer(text, return_tensors="pt")
inputs_ids = inputs.input_ids.to(model.device)

n_tokens_decode = 2
prefill_ids = inputs_ids[:, :-n_tokens_decode]
decode_id = inputs_ids[:, -1]
position_ids=torch.arange(inputs_ids.shape[1], device=model.device).unsqueeze(0)
print(f"inputs_ids: {inputs_ids}")
print(f"prefill_ids: {prefill_ids}")
print(f"decode_id: {decode_id}")
print(f"position_ids: {position_ids}")

inputs_ids: tensor([[    1,  1824,   349,   272,   668,  2650,   374,   744,   302,   264,
           484,  2689, 19082, 28804,   415]], device='cuda:0')
prefill_ids: tensor([[    1,  1824,   349,   272,   668,  2650,   374,   744,   302,   264,
           484,  2689, 19082]], device='cuda:0')
decode_id: tensor([415], device='cuda:0')
position_ids: tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14]],
       device='cuda:0')


In [ ]:
popular_experts = [
                (9, 5),
                (11, 2),
                (10, 4),
                (28, 0),
                (13, 1),
                (17, 7),
                (12, 1),
                (8, 6),
                (16, 1),
                (9, 0),
                (14, 5),
                (19, 5),
                (26, 2),
                (30, 7),
                (7, 1),
                (3, 7),
                (23, 4),
                (22, 1),
                (29, 3),
                (1, 5),
                (13, 0),
                (5, 1),
                (18, 0),
                (4, 7),
                (10, 3),
                (1, 2),
                (3, 0),
                (8, 3),
                (11, 0),
                (11, 5),
                (11, 1),
                (31, 4),
                (21, 0),
                (25, 1),
                (15, 5),
                (22, 4),
                (27, 5),
                (16, 7),
                (15, 1),
                (13, 2),
                (15, 4),
                (21, 1),
                (27, 7),
                (9, 7),
                (7, 4),
                (31, 5),
                (2, 1),
                (11, 6),
                (12, 3),
                (2, 4),
                (24, 2),
                (28, 2),
                (0, 2),
                (30, 2),
                (6, 0),
                (6, 7),
                (15, 6),
                (6, 2),
                (14, 2),
                (2, 0),
                (17, 2),
                (19, 2),
                (24, 0),
                (10, 0),
                (19, 4),
                (1, 4),
                (26, 3),
                (31, 7),
                (17, 6),
                (25, 3),
                (12, 6),
                (0, 0),
                (26, 0),
                (29, 7),
                (27, 2),
                (19, 6),
                (5, 0),
                (18, 2),
                (20, 1),
                (12, 4),
                (17, 5),
                (5, 4),
                (30, 6),
                (20, 5),
                (24, 6),
                (25, 2),
                (28, 4),
                (4, 6),
                (7, 2),
                (20, 3),
                (23, 2),
                (8, 4),
                (30, 0),
                (3, 4),
                (12, 5),
                (23, 7),
                (1, 7),
                (22, 5),
                (18, 4),
                (31, 0),
                (17, 0),
                (0, 5),
                (14, 6),
                (0, 3),
                (15, 7),
                (5, 6),
                (4, 4),
                (24, 7),
                (31, 1),
                (27, 6),
                (22, 2),
                (14, 1),
                (1, 0),
                (29, 1),
                (21, 3),
                (25, 7),
                (22, 3),
                (7, 3),
                (2, 6),
                (29, 5),
                (28, 3),
                (6, 6),
                (7, 5),
                (5, 7),
                (8, 5),
                (20, 4),
                (21, 5),
                (18, 7),
                (27, 0),
                (16, 0),
                (24, 5),
                (12, 2),
                (2, 2),
                (24, 3),
                (4, 1),
                (29, 0),
                (3, 1),
                (21, 6),
                (10, 2),
                (20, 7),
                (19, 0),
                (26, 7),
                (20, 6),
                (23, 3),
                (4, 3),
                (30, 1),
                (1, 6),
                (29, 2),
                (30, 3),
                (0, 6),
                (8, 1),
                (25, 6),
                (29, 4),
                (16, 2),
                (23, 1),
                (26, 1),
                (26, 6),
                (16, 4),
                (2, 5),
                (0, 4),
                (7, 6),
                (14, 4),
                (3, 6),
                (20, 0),
                (18, 3),
                (4, 5),
                (17, 4),
                (0, 1),
                (16, 5),
                (19, 3),
                (23, 0),
                (30, 4),
                (20, 2),
                (13, 6),
                (18, 6),
                (15, 2),
                (3, 5),
                (22, 0),
                (10, 1),
                (9, 6),
                (10, 5),
                (25, 4),
                (9, 2),
                (18, 1),
                (6, 4),
                (4, 2),
                (23, 5),
                (6, 5),
                (21, 2),
                (5, 5),
                (6, 1),
                (26, 5),
                (12, 0),
                (25, 0),
                (4, 0),
                (14, 0),
                (16, 6),
                (31, 2),
                (8, 0),
                (21, 7),
                (14, 3),
                (31, 6),
                (28, 1),
                (5, 3),
                (23, 6),
                (6, 3),
                (18, 5),
                (25, 5),
                (27, 1),
                (11, 7),
                (11, 4),
                (24, 1),
                (0, 7),
                (8, 7),
                (13, 3),
                (21, 4),
                (27, 4),
                (13, 7),
                (3, 2),
                (9, 1),
                (2, 7),
                (7, 0),
                (2, 3),
                (28, 5),
                (27, 3),
                (15, 0),
                (24, 4),
                (5, 2),
                (22, 6),
                (3, 3),
                (28, 6),
                (14, 7),
                (13, 4),
                (28, 7),
                (22, 7),
                (13, 5),
                (19, 1),
                (26, 4),
                (1, 1),
                (17, 1),
                (16, 3),
                (10, 7),
                (29, 6),
                (19, 7),
                (31, 3),
                (7, 7),
                (1, 3),
                (8, 2),
                (9, 4),
                (17, 3),
                (30, 5),
                (15, 3),
                (9, 3),
                (10, 6),
                (12, 7),
                (11, 3),
            ]
# 取前50个元素
popular_experts = popular_experts[:50]
# 排序
popular_experts.sort(key=lambda x: (x[0], x[1]))
# 打印排序后的列表
for item in popular_experts:
    print(item)

In [ ]:
n_layers = 32
n_experts = 8

start_layer = 2
end_layer = 30
act_experts = 2

(start_layer + 1 + n_layers - end_layer) * n_experts + (end_layer - start_layer - 1) * act_experts

94

In [ ]:
import pandas as pd
parquet_file = "/zx_data1/models/datasets/wikitext-tmp/train-00000-of-00002.parquet"
df = pd.read_parquet(parquet_file)
print(df.head())  # 打印前几行数据，查看列名和数据
print(df.columns) # 打印所有列名

                                                text
0                                                   
1                     = Valkyria Chronicles III = \n
2                                                   
3   Senjō no Valkyria 3 : <unk> Chronicles ( Japa...
4   The game began development in 2010 , carrying...
Index(['text'], dtype='object')


In [ ]:
import numpy as np
import os
# 验证保存结果
save_dir = "/zx_data1/sparsity/on_device_sd/log/logits"
model_name = "Mixtral-8x7B"

data = np.load(os.path.join(save_dir, f"{model_name}_probs.npz"), allow_pickle=True)
probs = data['probs']
prefix_tokens = data['prefix_tokens']
i = 1008
print(f"prefix_tokens: {prefix_tokens[i]}")
print(f"probs: {probs[i]}")
print(f"sum of probs: {np.sum(probs[i])}")
print(f"probs.shape: {probs[i].shape}")
print(f"prefix_tokens.shape: {len(prefix_tokens[i])}")

prefix_tokens: [1, 28705, 415, 2039, 464, 28713, 6651, 1587, 1200, 272, 28705, 0, 28705, 1587, 1200, 349, 7158, 754, 5090, 477, 28705, 0, 28705, 23967, 4992, 842, 6213, 23225, 1200, 5117, 5339, 1430, 5028, 1413, 264, 1830, 802, 28733, 28818, 1060, 10403, 302, 272, 6651, 2222, 3341, 714, 2327, 264, 3233, 349, 5937, 1200, 272, 4385, 11662, 272, 3233, 1401, 272, 6651, 2222, 297, 4008, 802, 28733, 28818, 1338, 842, 330, 3233, 541, 865, 960, 2327, 660, 802, 28733, 28818, 1527, 1200, 562, 6128, 541, 347, 12295, 5166, 8617, 438, 272, 19475, 302, 799, 6128, 464, 8617, 842, 7066, 3233, 659, 264, 1834, 304, 5328, 302, 6249, 6516, 486, 652, 9624, 420, 25793, 842, 4324, 298, 9542, 6128, 541, 347, 11400, 298, 264, 2692, 7023, 842, 6213, 2039, 1674, 1200, 6128, 622, 1034, 575, 513, 1545, 6881, 298, 706, 1200, 1259, 390, 652, 2528, 3569, 325, 22379, 1143, 2719, 2859, 442, 1250, 18625, 575, 486, 8454, 10813, 842, 7066, 3233, 659, 2948, 345, 10650, 10633, 345, 1200, 6266, 4842, 298, 1430, 3233, 842, 13